# EXP-14-A/B — CIFAR-10 ResNet-18 : PyTorch
**Objectif** : reproduire les résultats du Chapitre 14 sur CIFAR-10.

| Expérience | Méthode | Top-1 attendu | Gain comp. |
|------------|---------|--------------|------------|
| EXP-14-A | K-ABENA Adaptatif (SGD) | **94.9%** (+1.7%) | 19.3% |
| EXP-14-B | Adam + K-ABENA | **95.1%** (+1.6%) | 17.8% |

> **Coût adoption** : +2 lignes dans la boucle d'entraînement standard.

In [ ]:
# Installations
# !pip install kabena-ml[torch] torchvision -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.models as models
import numpy as np
from torch.utils.data import DataLoader

from kabena_ml.core.filter import calibrate_K
from kabena_ml.integrations.torch_utils import kabena_filter_torch

# ── Configuration ─────────────────────────────────────────────────────────
SEED      = 42
EPOCHS    = 200    # réduire à 20 pour test rapide
BATCH     = 128
LR        = 0.1
N_ABENA   = 0.3    # proportion de mineures CONSERVÉES
T_WARMUP  = 20     # époques de warm-up pour K adaptatif
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device : {DEVICE}')

In [ ]:
# ── Données CIFAR-10 ──────────────────────────────────────────────────────
tfm_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
tfm_test = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_set  = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=tfm_train)
test_set   = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=tfm_test)
train_load = DataLoader(train_set, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
test_load  = DataLoader(test_set,  batch_size=256,   shuffle=False, num_workers=2, pin_memory=True)

print(f'Train : {len(train_set)} | Test : {len(test_set)}')

In [ ]:
# ── Modèle ResNet-18 adapté CIFAR-10 ─────────────────────────────────────
# ResNet-18 modifié : conv1 3×3 (au lieu de 7×7), sans MaxPool initial
def resnet18_cifar():
    m = models.resnet18(weights=None, num_classes=10)
    m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()  # supprime le MaxPool initial
    return m

# ── Évaluation ────────────────────────────────────────────────────────────
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(1)
            correct += (preds == y).sum().item()
            total   += y.size(0)
    return correct / total * 100

# ── Scheduler cosine + warmup ─────────────────────────────────────────────
def get_scheduler(optimizer, epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

---
## EXP-14-A : K-ABENA Adaptatif (SGD)
**Coût : +2 lignes** dans la boucle `for X, y in loader`

In [ ]:
model_A = resnet18_cifar().to(DEVICE)
opt_A   = torch.optim.SGD(model_A.parameters(), lr=LR,
                           momentum=0.9, weight_decay=1e-4, nesterov=True)
sched_A = get_scheduler(opt_A, EPOCHS)

# K adaptatif : warm-up progressif de 5% → 20% sur T_WARMUP époques
K_t     = None   # sera calibré à l'époque 1
history_A = []

for epoch in range(EPOCHS):
    model_A.train()
    ep_losses, ep_m, ep_n = [], 0, 0

    for X, y in train_load:
        X, y = X.to(DEVICE), y.to(DEVICE)

        # ── K-ABENA : 2 lignes de différence avec SGD standard ──────────
        losses = F.cross_entropy(model_A(X), y, reduction='none')  # ← 'none'
        mask   = kabena_filter_torch(losses, K=K_t or 1e9, N=N_ABENA)  # ← +1 ligne
        # ────────────────────────────────────────────────────────────────
        if not mask.any(): continue
        L_KA = losses[mask].mean()

        opt_A.zero_grad()
        L_KA.backward()
        opt_A.step()

        ep_losses.extend(losses.detach().cpu().numpy())
        ep_m += mask.sum().item()
        ep_n += y.size(0)

    sched_A.step()

    # Calibrage adaptatif de K (percentile croissant)
    q_t = 5 + (20 - 5) * min(epoch / T_WARMUP, 1.0)
    K_t = float(np.percentile(ep_losses, q_t))

    gain = round((1 - ep_m / ep_n) * 100)
    acc  = evaluate(model_A, test_load, DEVICE)
    history_A.append({'epoch': epoch, 'K': K_t, 'gain': gain, 'acc': acc})

    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        print(f'[EXP-14-A] Ep {epoch:3d} | K={K_t:.4f} | gain={gain}% | acc={acc:.2f}%')

acc_A = history_A[-1]['acc']
gain_A = np.mean([r['gain'] for r in history_A])
print(f'\n✓ EXP-14-A final : acc={acc_A:.2f}% | gain moyen={gain_A:.1f}%')
print(f'  Résultat attendu chapitre 14 : 94.9% | Gain : 19.3%')

---
## EXP-14-B : Adam + K-ABENA
**Même boucle, optimiseur Adam**

In [ ]:
model_B = resnet18_cifar().to(DEVICE)
opt_B   = torch.optim.Adam(model_B.parameters(), lr=1e-3, weight_decay=1e-4)
sched_B = torch.optim.lr_scheduler.CosineAnnealingLR(opt_B, T_max=EPOCHS)

K_t_B   = None
history_B = []

for epoch in range(EPOCHS):
    model_B.train()
    ep_losses, ep_m, ep_n = [], 0, 0

    for X, y in train_load:
        X, y = X.to(DEVICE), y.to(DEVICE)

        # ── Adam + K-ABENA : strictement même code que EXP-14-A ─────────
        losses = F.cross_entropy(model_B(X), y, reduction='none')  # ← 'none'
        mask   = kabena_filter_torch(losses, K=K_t_B or 1e9, N=N_ABENA)  # ← +1 ligne
        # ────────────────────────────────────────────────────────────────
        if not mask.any(): continue
        L_KA = losses[mask].mean()

        opt_B.zero_grad()
        L_KA.backward()
        opt_B.step()

        ep_losses.extend(losses.detach().cpu().numpy())
        ep_m += mask.sum().item()
        ep_n += y.size(0)

    sched_B.step()
    q_t    = 5 + (20 - 5) * min(epoch / T_WARMUP, 1.0)
    K_t_B  = float(np.percentile(ep_losses, q_t))
    gain   = round((1 - ep_m / ep_n) * 100)
    acc    = evaluate(model_B, test_load, DEVICE)
    history_B.append({'epoch': epoch, 'K': K_t_B, 'gain': gain, 'acc': acc})

    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        print(f'[EXP-14-B] Ep {epoch:3d} | K={K_t_B:.4f} | gain={gain}% | acc={acc:.2f}%')

print(f'\n✓ EXP-14-B final : acc={history_B[-1]["acc"]:.2f}%')
print(f'  Résultat attendu chapitre 14 : 95.1%')

In [ ]:
# ── Comparaison et visualisation ──────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep_A = [r['epoch'] for r in history_A]
ep_B = [r['epoch'] for r in history_B]

axes[0].plot(ep_A, [r['acc'] for r in history_A], color='#1DD0FF', lw=2, label='K-ABENA Adaptatif (SGD)')
axes[0].plot(ep_B, [r['acc'] for r in history_B], color='#03DC92', lw=2, label='Adam + K-ABENA')
axes[0].axhline(93.2, color='#B4B2A9', lw=1.5, ls='--', label='SGD Baseline (93.2%)')
axes[0].set_xlabel('Époque'); axes[0].set_ylabel('Accuracy test (%)')
axes[0].set_title('CIFAR-10 ResNet-18 — EXP-14-A et EXP-14-B'); axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.2)

axes[1].plot(ep_A, [r['K'] for r in history_A], color='#1DD0FF', lw=2, label='K adaptatif')
axes[1].plot(ep_A, [r['gain'] for r in history_A], color='#EF9F27', lw=2, ls='--', label='Gain (%)')
axes[1].set_xlabel('Époque'); axes[1].set_ylabel('Valeur')
axes[1].set_title('Trajectoire K adaptatif et gain comp.'); axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.2)

fig.suptitle('Reproduction EXP-14-A/B — Chapitre 14 K-ABENA', fontweight='bold')
plt.tight_layout()
plt.savefig('exp14ab_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n=== Résumé EXP-14-A/B ===')
print(f'EXP-14-A | acc={acc_A:.2f}% | gain={gain_A:.1f}% | attendu: 94.9% / 19.3%')
print(f'EXP-14-B | acc={history_B[-1]["acc"]:.2f}%           | attendu: 95.1% / 17.8%')